# 02c — Traduire `bau_bau` en parallèle

Ce notebook réutilise le moteur validé de `02b_traduire_mode_texte_playwright.ipynb`, mais avec une configuration totalement séparée pour la direction `bau_bau`.

Il traduit `instruction_fr → instruction_bau`, `input_fr → input_bau` lorsqu'il est non vide, et `output_fr → output_bau`. Les résultats sont stockés dans `data/02c_bau_bau_browser/`.

> Lancez ce notebook dans un **kernel distinct** de celui de `02b`. Les deux navigateurs utilisent chacun un délai de 16 secondes afin de conserver approximativement le débit agrégé du pipeline unique précédemment validé.

In [2]:
from pathlib import Path
import json
import os
import re

PROJECT_ROOT_02C = Path.cwd()
BASE_NOTEBOOK_PATH = PROJECT_ROOT_02C / "02b_traduire_fr_bau_texte_playwright.ipynb"

os.environ["LREE_PIPELINE_DIRECTION"] = "bau_bau"
os.environ["LREE_STEP_DIR"] = str(PROJECT_ROOT_02C / "data" / "02c_bau_bau_browser")
os.environ["LREE_DELAY_SECONDS"] = "16.0"
os.environ["LREE_INITIAL_DELAY_SECONDS"] = "8.0"

base_notebook = json.loads(BASE_NOTEBOOK_PATH.read_text(encoding="utf-8"))
base_cells = {cell.get("id"): cell for cell in base_notebook["cells"]}

def run_base_cell(cell_id):
    cell = base_cells[cell_id]
    code = "".join(cell["source"])
    exec(compile(code, f"{BASE_NOTEBOOK_PATH.name}:{cell_id}", "exec"), globals())

print("Notebook moteur :", BASE_NOTEBOOK_PATH)
print("Direction       : bau_bau")
print("Sorties         :", os.environ["LREE_STEP_DIR"])

Notebook moteur : /home/vedem_worker/scrapt-bau/02b_traduire_fr_bau_texte_playwright.ipynb
Direction       : bau_bau
Sorties         : /home/vedem_worker/scrapt-bau/data/02c_bau_bau_browser


## 1. Préparer exclusivement la file `bau_bau`

Cette étape charge la configuration, transforme les trois champs français nécessaires et construit des blocs indépendants de ceux de `fr_bau`.

In [3]:
for cell_id in ("cd8832ee", "7b299ae0", "1862faa4"):
    run_base_cell(cell_id)

assert PIPELINE_DIRECTION == "bau_bau"
assert set(FIELD_MAP) == {"bau_bau"}
assert all(unit["direction"] == "bau_bau" for unit in units)
print("✓ File bau_bau isolée et validée")

Répertoire de travail : /home/vedem_worker/scrapt-bau/data/02c_bau_bau_browser
Direction              : bau_bau
Code cible Google     : bci
Contrôle automatique tous les blocs : 100
Unités créées : 50,973
Fichier         : /home/vedem_worker/scrapt-bau/data/02c_bau_bau_browser/translation_units.jsonl
Blocs préparés       : 2,652
Caractères avec identifiants : 8,798,867
Contrôle automatique tous les : 100 blocs
Attente minimale entre contrôles : 26.7 minutes
✓ File bau_bau isolée et validée


## 2. Préparer Chromium

Cette cellule réutilise Chromium et les bibliothèques locales déjà installés pendant l'étape `02b`.

In [4]:
run_base_cell("59834e33")
run_base_cell("1cd9faea")

Playwright est déjà installé.
Vérification/installation du navigateur Chromium compatible…
Chromium Playwright est prêt.
Bibliothèques Chromium : /home/vedem_worker/scrapt-bau/.playwright-libs/usr/lib/x86_64-linux-gnu
Existe : True


## 3. Lancer la traduction continue `bau_bau`

Le navigateur s'ouvre une seule fois. Vérifiez **Français → Baoulé**, puis revenez au notebook et appuyez sur Entrée. Les contrôles sont exécutés automatiquement tous les 100 blocs et à chaque arrêt.

### Correction en cas d'erreur lors de traductions 
### (pour traduction baoulé baoulé mais passe aussi en cas d'erreur pour fr_bau) 

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import shutil

STEP_DIR = Path.cwd() / "data" / "02c_bau_bau_browser"
RESULTS_PATH = STEP_DIR / "browser_results.jsonl"
REPORT_PATH = STEP_DIR / "validation_report.json"

report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
errors = report.get("validation_errors", [])

if not errors:
    print("✓ Aucune erreur à réparer.")
else:
    bad_batch_ids = {
        error.get("batch_id")
        for error in errors
    }

    print("Blocs à retraduire :", sorted(map(str, bad_batch_ids)))

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_path = STEP_DIR / f"browser_results_backup_{timestamp}.jsonl"
    shutil.copy2(RESULTS_PATH, backup_path)

    records = []
    with RESULTS_PATH.open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                records.append(json.loads(line))

    kept_records = [
        record
        for record in records
        if record.get("batch_id") not in bad_batch_ids
    ]

    removed_records = len(records) - len(kept_records)

    temporary_path = RESULTS_PATH.with_suffix(".jsonl.tmp")
    with temporary_path.open("w", encoding="utf-8", newline="\n") as handle:
        for record in kept_records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")

    temporary_path.replace(RESULTS_PATH)

    print(f"✓ {removed_records} résultat(s) invalide(s) retiré(s)")
    print("✓ Sauvegarde créée :", backup_path)
    print("✓ Les autres traductions ont été conservées")

In [ ]:
run_base_cell("d3ff9847")

### EXECUTION AUTONOME RAPIDE BAU_BAU — exécuter le code ci-dessous à la place du lanceur pour le mode en parallelle ci-dessus


In [8]:
# Le mode parallèle historique reste inchangé dans les cellules précédentes.
from collections import defaultdict
from datetime import datetime
from pathlib import Path
import builtins
import json
import os
import shutil

AUTO_PROJECT_ROOT = Path.cwd()
AUTO_BASE_NOTEBOOK = AUTO_PROJECT_ROOT / "02b_traduire_fr_bau_texte_playwright.ipynb"
if not AUTO_BASE_NOTEBOOK.exists():
    AUTO_BASE_NOTEBOOK = AUTO_PROJECT_ROOT / "02b_traduire_mode_texte_playwright.ipynb"
if not AUTO_BASE_NOTEBOOK.exists():
    raise FileNotFoundError("Notebook moteur 02b introuvable.")

# Une seule traduction tourne maintenant : cadence équivalente au débit agrégé prudent antérieur.
os.environ["LREE_PIPELINE_DIRECTION"] = "bau_bau"
os.environ["LREE_STEP_DIR"] = str(AUTO_PROJECT_ROOT / "data" / "02c_bau_bau_browser")
os.environ["LREE_DELAY_SECONDS"] = "8.0"
os.environ["LREE_INITIAL_DELAY_SECONDS"] = "0.0"

auto_notebook = json.loads(AUTO_BASE_NOTEBOOK.read_text(encoding="utf-8"))
auto_cells = {cell.get("id"): cell for cell in auto_notebook["cells"]}

def run_auto_base_cell(cell_id):
    cell = auto_cells[cell_id]
    code = "".join(cell["source"])
    exec(compile(code, f"{AUTO_BASE_NOTEBOOK.name}:{cell_id}", "exec"), globals())

# Recharge toute la configuration bau_bau afin que cette cellule soit autonome.
for auto_cell_id in ("cd8832ee", "7b299ae0", "1862faa4", "1cd9faea"):
    run_auto_base_cell(auto_cell_id)

assert PIPELINE_DIRECTION == "bau_bau"
assert all(unit["direction"] == "bau_bau" for unit in units)
DELAY_SECONDS = 8.0
INITIAL_DELAY_SECONDS = 0.0

# Charge les fonctions du moteur sans lancer l'ancien point d'entrée.
RUN_BROWSER = False
run_auto_base_cell("d3ff9847")
RUN_BROWSER = True

AUTO_MAX_RETRIES_PER_BATCH = 3
auto_retry_counts = defaultdict(int)
AUTO_MARKER_PATTERN = re.compile(
    r"^[ \t]*(\d{8}):[ \t]*(.*?)(?=^[ \t]*\d{8}:|\Z)",
    flags=re.DOTALL | re.MULTILINE,
)
AUTO_STRUCTURAL_EMPTY_PATTERN = re.compile(
    r"^(?:\s|<br\s*/?>|&nbsp;)*$",
    flags=re.IGNORECASE,
)

def repair_invalid_results(report):
    errors = report.get("validation_errors", [])
    if not errors:
        return []

    bad_batch_ids = {error.get("batch_id") for error in errors}
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    backup_path = STEP_DIR / f"browser_results_backup_auto_{timestamp}.jsonl"
    shutil.copy2(RESULTS_PATH, backup_path)

    records = read_jsonl(RESULTS_PATH) if RESULTS_PATH.exists() else []
    batch_lookup = {batch["batch_id"]: batch for batch in batches}
    result_positions = defaultdict(list)
    for position, record in enumerate(records):
        if record.get("status") == "ok":
            result_positions[record.get("batch_id")].append(position)

    # Google supprime parfois les entrées purement structurelles comme <br>.
    # Elles ne contiennent aucun texte à traduire : on conserve le marqueur source
    # pour terminer la validation, puis il sera normalisé en chaîne vide au merge final.
    structural_batch_ids = set()
    for error in errors:
        batch_id = error.get("batch_id")
        positions = result_positions.get(batch_id, [])
        batch = batch_lookup.get(batch_id)
        if error.get("error") != "empty_translation" or not positions or batch is None:
            continue

        result = records[positions[-1]]
        matches = [
            (f"u{numeric_id}", text.strip())
            for numeric_id, text in AUTO_MARKER_PATTERN.findall(result["translated_text"])
        ]
        if [unit_id for unit_id, _ in matches] != batch["unit_ids"]:
            continue

        empty_unit_ids = [unit_id for unit_id, text in matches if not text]
        if not empty_unit_ids or not all(
            AUTO_STRUCTURAL_EMPTY_PATTERN.fullmatch(unit_by_id[unit_id]["source_text"])
            for unit_id in empty_unit_ids
        ):
            continue

        patched_text = "\n".join(
            f"{unit_id[1:]}: {text or unit_by_id[unit_id]['source_text']}"
            for unit_id, text in matches
        )
        for position in positions:
            records[position]["translated_text"] = patched_text
            records[position]["structural_empty_normalized"] = True
        structural_batch_ids.add(batch_id)
        print(f"✓ {batch_id}: {empty_unit_ids} contient uniquement <br> ; traduction non requise")

    retry_batch_ids = bad_batch_ids - structural_batch_ids
    for batch_id in retry_batch_ids:
        auto_retry_counts[batch_id] += 1
        if auto_retry_counts[batch_id] > AUTO_MAX_RETRIES_PER_BATCH:
            raise RuntimeError(
                f"{batch_id} reste invalide après {AUTO_MAX_RETRIES_PER_BATCH} tentatives ; "
                "arrêt pour contrôle manuel."
            )

    kept_records = [record for record in records if record.get("batch_id") not in retry_batch_ids]
    removed_count = len(records) - len(kept_records)
    write_jsonl_atomic(RESULTS_PATH, kept_records)

    if retry_batch_ids:
        print("Blocs remis en traduction :", sorted(map(str, retry_batch_ids)))
        print(f"✓ {removed_count} résultat(s) invalide(s) mis de côté")
    if structural_batch_ids:
        print("Blocs structurels corrigés sans retraduction :", sorted(structural_batch_ids))
    print("✓ Sauvegarde :", backup_path)

    repaired_report = parse_and_validate_results(verbose=True)
    if repaired_report["validation_error_count"]:
        raise RuntimeError("Des erreurs subsistent après la réparation automatique.")
    return sorted(map(str, bad_batch_ids))

def run_autonomous_bau_bau():
    print("=== Mode autonome bau_bau ===")
    print(f"Délai entre blocs : {DELAY_SECONDS:.0f} secondes")
    print("Décalage initial   : désactivé")

    while True:
        report = parse_and_validate_results(verbose=True)
        if report["validation_error_count"]:
            repair_invalid_results(report)

        if len(completed_batch_ids()) >= len(batches):
            print("✓ Toutes les traductions bau_bau sont terminées et validées.")
            return

        # Les redémarrages dus au contrôle ne nécessitent plus une validation manuelle au clavier.
        original_input = builtins.input
        builtins.input = lambda prompt="": (print(prompt + "validation automatique"), "")[1]
        try:
            run_coroutine_in_thread(run_browser_session())
        finally:
            builtins.input = original_input

        report = parse_and_validate_results(verbose=True)
        if report["validation_error_count"]:
            repair_invalid_results(report)
            print("✓ Reprise automatique de la traduction…")
            continue

        if len(completed_batch_ids()) >= len(batches):
            print("✓ Toutes les traductions bau_bau sont terminées et validées.")
        else:
            print("Session arrêtée sans erreur de contrôle. Consultez le dernier message avant de relancer.")
        return

try:
    run_autonomous_bau_bau()
finally:
    # Empêche le lanceur standard situé juste après de démarrer une seconde session avec « Exécuter tout ».
    RUN_BROWSER = False

Répertoire de travail : /home/vedem_worker/scrapt-bau/data/02c_bau_bau_browser
Direction              : bau_bau
Code cible Google     : bci
Contrôle automatique tous les blocs : 100
Unités créées : 50,973
Fichier         : /home/vedem_worker/scrapt-bau/data/02c_bau_bau_browser/translation_units.jsonl
Blocs préparés       : 2,652
Caractères avec identifiants : 8,798,867
Contrôle automatique tous les : 100 blocs
Attente minimale entre contrôles : 13.3 minutes
Bibliothèques Chromium : /home/vedem_worker/scrapt-bau/.playwright-libs/usr/lib/x86_64-linux-gnu
Existe : True
Navigateur non lancé. Passez RUN_BROWSER à True dans la première cellule après avoir lu les consignes.
=== Mode autonome bau_bau ===
Délai entre blocs : 8 secondes
Décalage initial   : désactivé
--- Contrôle automatique ---
Blocs validés       : 2,652/2,652
Fragments analysés  : 50,828
Champs reconstruits : 50,821
Erreurs de contrôle : 7
Sortie partielle     : /home/vedem_worker/scrapt-bau/data/02c_bau_bau_browser/partial_t

## 4. Relancer uniquement le contrôle (optionnel)

Cette cellule reconstruit `partial_translations.jsonl` sans ouvrir le navigateur.

In [9]:
validation_report = parse_and_validate_results(verbose=True)

--- Contrôle automatique ---
Blocs validés       : 2,652/2,652
Fragments analysés  : 50,973
Champs reconstruits : 50,966
Erreurs de contrôle : 0
Sortie partielle     : /home/vedem_worker/scrapt-bau/data/02c_bau_bau_browser/partial_translations.jsonl
